In [19]:
import polars as pl
import json
from pathlib import Path

In [20]:

PROJECT_ROOT = Path().resolve().parents[1]

df = pl.read_parquet(
    PROJECT_ROOT / "data" / "clean" / "flights_clean.parquet"
)

# count flights between each airport pair
# this becomes edge weights in the graph
route_counts = (
    df.filter(pl.col("COVID_FLAG") == 0)
    .group_by(["ORIGIN", "DEST"])
    .agg(pl.len().alias("flight_count"))
    .sort("flight_count", descending=True)
)

print(route_counts.shape)
print(route_counts.head(20))

(7484, 3)
shape: (20, 3)
┌────────┬──────┬──────────────┐
│ ORIGIN ┆ DEST ┆ flight_count │
│ ---    ┆ ---  ┆ ---          │
│ str    ┆ str  ┆ u32          │
╞════════╪══════╪══════════════╡
│ SFO    ┆ LAX  ┆ 3451         │
│ LAX    ┆ SFO  ┆ 3336         │
│ LGA    ┆ ORD  ┆ 3287         │
│ ORD    ┆ LGA  ┆ 3180         │
│ OGG    ┆ HNL  ┆ 3118         │
│ …      ┆ …    ┆ …            │
│ ATL    ┆ MCO  ┆ 2270         │
│ LGA    ┆ ATL  ┆ 2258         │
│ PHX    ┆ DEN  ┆ 2235         │
│ DEN    ┆ PHX  ┆ 2219         │
│ ATL    ┆ LGA  ┆ 2201         │
└────────┴──────┴──────────────┘


In [21]:
import sys
!{sys.executable} -m pip install networkx
import networkx as nx

# install if needed: pip install networkx
G = nx.DiGraph()

# filter to only routes involving your scored airports
# and routes with meaningful frequency (at least 500 flights)
SCORED_AIRPORTS = [
    "ATL", "DFW", "ORD", "DEN", "CLT",
    "LAX", "LAS", "LGA", "SEA", "PHX"
]

# add all routes as edges, weight = flight frequency
for row in route_counts.iter_rows(named=True):
    origin = row["ORIGIN"]
    dest = row["DEST"]
    count = row["flight_count"]
    if count >= 500:  # only meaningful routes
        G.add_edge(origin, dest, weight=count)

print(f"nodes: {G.number_of_nodes()}")
print(f"edges: {G.number_of_edges()}")

# check connectivity of your scored airports
print("\nscored airport connections:")
for airport in SCORED_AIRPORTS:
    if airport in G:
        out_degree = G.out_degree(airport, weight="weight")
        in_degree = G.in_degree(airport, weight="weight")
        print(f"{airport}: out={out_degree:,} in={in_degree:,}")

nodes: 127
edges: 1137

scored airport connections:
ATL: out=74,935 in=74,611
DFW: out=47,600 in=46,917
ORD: out=52,009 in=52,263
DEN: out=48,976 in=49,875
CLT: out=34,851 in=35,604
LAX: out=45,391 in=43,418
LAS: out=33,552 in=34,203
LGA: out=33,552 in=33,277
SEA: out=30,148 in=30,324
PHX: out=31,745 in=33,520



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
# convert raw flight counts to propagation probability
# higher frequency route = stronger cascade connection
# normalize per origin airport so probabilities sum to 1

propagation = {}

for airport in G.nodes():
    neighbors = list(G.successors(airport))
    if not neighbors:
        continue
    
    total_weight = sum(
        G[airport][n]["weight"] for n in neighbors
    )
    
    propagation[airport] = {
        n: G[airport][n]["weight"] / total_weight
        for n in neighbors
    }

# check ATL's top downstream airports
atl_downstream = sorted(
    propagation["ATL"].items(),
    key=lambda x: x[1],
    reverse=True
)[:10]

print("ATL top 10 downstream:")
for dest, prob in atl_downstream:
    print(f"  {dest}: {prob:.3f}")

ATL top 10 downstream:
  MCO: 0.030
  LGA: 0.029
  FLL: 0.029
  MIA: 0.025
  DCA: 0.024
  TPA: 0.024
  DFW: 0.023
  ORD: 0.022
  BWI: 0.022
  EWR: 0.021


In [26]:
def simulate_cascade(
    stressed_airport: str,
    current_scores: dict,
    propagation: dict,
    threshold: float = 70.0,
    hours_ahead: int = 6,
    decay: float = 0.6
) -> dict:
    """
    When an airport crosses threshold, simulate downstream impact.
    
    decay: how much impact reduces per hop (0.6 = 40% reduction per hop)
    """
    
    results = {h: {} for h in range(1, hours_ahead + 1)}
    
    # only cascade if airport is actually stressed
    trigger_score = current_scores.get(stressed_airport, {})
    if isinstance(trigger_score, dict):
        trigger_score = trigger_score.get("score", 0)
    
    if trigger_score < threshold:
        return results
    
    # impact strength = how far above threshold
    initial_impact = max(
    (trigger_score - threshold) / (100 - threshold),
    0.1  # minimum impact so borderline airports still cascade
)
    
    # BFS through the graph - propagate impact hop by hop
    visited = {stressed_airport: initial_impact}
    current_wave = {stressed_airport: initial_impact}
    
    for hour in range(1, hours_ahead + 1):
        next_wave = {}
        
        for airport, impact in current_wave.items():
            if airport not in propagation:
                continue
                
            for downstream, prob in propagation[airport].items():
                if downstream in visited:
                    continue
                    
                # impact = upstream impact × route probability × decay
                downstream_impact = impact * prob * decay * 20
                
                if downstream_impact > 0.01:  # ignore tiny impacts
                    if downstream not in next_wave:
                        next_wave[downstream] = 0
                    next_wave[downstream] += downstream_impact
                    visited[downstream] = downstream_impact
        
        # store impacts for this hour
        results[hour] = {
            airport: round(min(impact * 100, 50.0), 1)
            for airport, impact in next_wave.items()
        }
        current_wave = next_wave
    
    return results

In [28]:


PROJECT_ROOT = Path().resolve().parents[1]

# load latest live scores
with open(PROJECT_ROOT / "data" / "clean" / "live_scores.json") as f:
    live_scores = json.load(f)

print("current scores:")
for airport, data in sorted(
    live_scores.items(), 
    key=lambda x: x[1]["score"], 
    reverse=True
):
    print(f"  {airport}: {data['score']}")

# simulate cascade from highest scoring airport
top_airport = max(live_scores.items(), key=lambda x: x[1]["score"])[0]
print(f"\nsimulating cascade from {top_airport} (score: {live_scores[top_airport]['score']})")

cascade = simulate_cascade(top_airport, live_scores, propagation, threshold=65.0)

print("\n--- cascade impact by hour ---")
for hour, impacts in cascade.items():
    if impacts:
        print(f"\nT+{hour}h:")
        for airport, impact in sorted(impacts.items(), key=lambda x: x[1], reverse=True)[:5]:
            print(f"  {airport}: +{impact} congestion impact")

current scores:
  DFW: 70.0
  SEA: 58.5
  PHX: 49.2
  ORD: 48.9
  DEN: 47.1
  ATL: 46.7
  LAS: 31.6
  YVR: 27.8
  LAX: 26.2
  LGA: 24.3
  CLT: 20.0
  YOW: 7.0

simulating cascade from DFW (score: 70.0)

--- cascade impact by hour ---

T+1h:
  LAX: +6.5 congestion impact
  ORD: +6.3 congestion impact
  LGA: +6.1 congestion impact
  ATL: +6.1 congestion impact
  DEN: +5.4 congestion impact

T+2h:
  SJU: +2.9 congestion impact
  ANC: +2.9 congestion impact
  HNL: +2.6 congestion impact
  SJC: +2.5 congestion impact
  SMF: +2.2 congestion impact

T+3h:
  FAI: +7.5 congestion impact
  KOA: +5.8 congestion impact
  LIH: +5.5 congestion impact
  ITO: +4.2 congestion impact
  LGB: +1.4 congestion impact


In [29]:
import pickle

# save networkx graph
with open(PROJECT_ROOT / "models" / "route_graph.pkl", "wb") as f:
    pickle.dump(G, f)

# save propagation probabilities
with open(PROJECT_ROOT / "models" / "propagation.pkl", "wb") as f:
    pickle.dump(propagation, f)

print("saved route_graph.pkl and propagation.pkl")

saved route_graph.pkl and propagation.pkl


In [30]:
# run cascade for all currently stressed airports (score > 65)
all_cascades = {}

for airport, data in live_scores.items():
    score = data["score"] if isinstance(data, dict) else 0
    if score >= 65:
        cascade = simulate_cascade(
            airport,
            live_scores,
            propagation,
            threshold=65.0
        )
        all_cascades[airport] = cascade
        print(f"{airport} ({score}) → cascading")

# save
with open(PROJECT_ROOT / "data" / "clean" / "cascade_forecast.json", "w") as f:
    json.dump(all_cascades, f, indent=2)

print(f"\nsaved cascade_forecast.json")
print(f"triggering airports: {list(all_cascades.keys())}")

DFW (70.0) → cascading

saved cascade_forecast.json
triggering airports: ['DFW']
